# 11. Fine-tuning V2 с GroupKFold

Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


## Исправление исторической утечки holdout

Перед V2 сохраняем групповой FT-split: спектры одного соединения больше не
разделяются между train и validation. Сам V2 далее использует 5-fold GroupKFold.

In [ ]:
import subprocess

def run_script(script_name, *args):

    # Run one reproducible pipeline stage and stream its output.
    
    command = [sys.executable, str(PROJECT_ROOT / "src" / script_name), *map(str, args)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [3]:
run_script("resplit_ft.py")

Running: d:\Dev\envs\study\python.exe D:\Users\user\Desktop\DS_XRD_project\src\resplit_ft.py


# Fine-tuning v2: большая модель + 5-fold group CV

Отличия от v1:
- старт с `pretrain_v2_full_best.pt` (11.3M);
- **устойчивая оценка**: 5-fold кросс-валидация с группировкой по соединению
  (conn_key = фаза + решётка; соединение никогда не бывает в двух фолдах);
- метрики = среднее ± std по 5 фолдам + агрегат по всем фолдам (полный val);
- финальная модель: обучение на всём FT-пуле с числом эпох = медиана лучших
  эпох фолдов → `ft_v2_final.pt`;
- повышенный вес углов решётки (как в пре-трейне v2).

Время: 5 фолдов × 25 эпох + финал ≈ 10–15 мин на 3060 Ti.

In [ ]:
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '|', torch.cuda.get_device_name(0))

device: cuda | NVIDIA GeForce RTX 3060 Ti


In [ ]:
MODE = 'cv'   # 'cv' - 5 фолдов + финальная модель; 'quick' - 1 фолд для отладки

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
CKPT_PRE = BASE / 'checkpoints' / 'pretrain_v2_full_best.pt'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'

GRID_N = 4096
W = 3
SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']
IMPUTE_LAMBDA = 1.5406
W_LAT, W_VOL, W_SG, W_SYS, W_EL, W_ANG = 1.0, 0.5, 1.0, 1.0, 1.0, 2.0

HP = dict(
    batch=128,
    lr_head=3e-5,
    lr_backbone=1e-5,
    wd=1e-4,
    clip=1.0,
    epochs=50,
    replay_frac=0.5,
    warmup_frac=0.1,
    n_folds=5,
)

In [ ]:
stats = json.loads((OUT / 'pretrain_stats.json').read_text())
VOCAB = stats['vocab']
EL_IDX = {e: i for i, e in enumerate(VOCAB)}
LAT_MEAN = np.array(stats['lat_mean'])
LAT_STD = np.array(stats['lat_std'])
VOL_MEAN, VOL_STD = stats['vol_mean'], stats['vol_std']

index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
N_TOTAL = len(index)
X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r', shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r', shape=(N_TOTAL, GRID_N))
row_of = dict(zip(index['sample_id'], index['row_idx']))

ft = pd.read_parquet(BASE / 'data' / 'clean' / 'ft_pool_combined.parquet')
ft['row_idx'] = ft['sample_id'].map(row_of)
assert ft['row_idx'].notna().all()
ft['primary_wavelength'] = ft['primary_wavelength'].fillna(IMPUTE_LAMBDA)

abc = ft[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang = ft[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
ft['V'] = abc.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
ft['lat6'] = list(np.hstack([np.log(abc), ang]))

def to_list(v):
    if isinstance(v, str):
        try:
            return json.loads(v)
        except Exception:
            return []
    if isinstance(v, (list, tuple, np.ndarray)):
        return [e for e in v if isinstance(e, str)]
    return []

ft['elements'] = ft['elements_list'].apply(to_list)
ft['conn_key'] = (ft['phase_compositions'].astype(str) + '|'
                  + ft['lattice_a'].round(2).astype(str))
print('FT-пул:', len(ft), '| уникальных соединений:', ft['conn_key'].nunique())

# replay-пул (трейн-часть синтетики, как в v1)
splits_pre = pd.read_parquet(OUT / 'splits_pretrain.parquet')
syn_meta = pd.read_parquet(BASE / 'data' / 'clean' / 'df_synth_summary_final_clean.parquet')[
    ['sample_id', 'lattice_a', 'lattice_b', 'lattice_c',
     'alpha', 'beta', 'gamma', 'spacegroup_number', 'crystal_system', 'elements_list']
]
pre_rows = index[index['split_role'] == 'pretrain'][['sample_id', 'row_idx',
                                                      'lambda_1', 'lambda_2']]
syn = pre_rows.merge(splits_pre, on='sample_id').merge(syn_meta, on='sample_id')
syn = syn[syn['split'] == 'train'].reset_index(drop=True)
abc_s = syn[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang_s = syn[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang_s[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
syn['V'] = abc_s.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
syn['lat6'] = list(np.hstack([np.log(abc_s), ang_s]))
syn['elements'] = syn['elements_list'].apply(to_list)
print('replay-пул:', len(syn))

FT-пул: 3475 | уникальных соединений: 2029
replay-пул: 451027


In [ ]:
# ---------- датасеты и лоадеры ----------
def build_labels(frame, is_real):
    lam1_col = 'primary_wavelength' if 'primary_wavelength' in frame else 'lambda_1'
    lam2_col = 'secondary_wavelength' if 'secondary_wavelength' in frame else 'lambda_2'
    lam1 = frame[lam1_col].to_numpy(np.float32)
    lam2 = frame[lam2_col].to_numpy(np.float32)
    lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54,
                    np.isfinite(lam2).astype(np.float32)], 1)
    lat6 = np.stack(frame['lat6'].to_numpy())
    has_lat = ~np.isnan(lat6).any(1)
    latm = has_lat.astype(np.float32)
    lat6 = (lat6 - LAT_MEAN) / LAT_STD
    vol = (np.log(frame['V'].to_numpy(np.float64)) - VOL_MEAN) / VOL_STD
    sg_raw = frame['spacegroup_number'].to_numpy(float)
    sgm = np.isfinite(sg_raw).astype(np.float32)
    sg = np.nan_to_num(sg_raw).astype(np.int64) - 1
    sysmap = {s: i for i, s in enumerate(SYSTEMS)}
    sys_raw = frame['crystal_system'].map(sysmap)
    sysm = sys_raw.notna().to_numpy(np.float32)
    sys_ = sys_raw.fillna(0).to_numpy(np.int64)
    n = len(frame)
    el = np.zeros((n, len(VOCAB)), np.float32)
    elm = np.zeros(n, np.float32)
    for i, els in enumerate(frame['elements']):
        if len(els):
            elm[i] = 1.0
            for e in els:
                j = EL_IDX.get(e)
                if j is not None:
                    el[i, j] = 1.0
    return dict(lam=lam, lat6=lat6.astype(np.float32), latm=latm,
                vol=vol.astype(np.float32), volm=latm.copy(),
                sg=sg, sgm=sgm, sys_=sys_, sysm=sysm, el=el, elm=elm)

class SpecDS(Dataset):
    def __init__(self, frame, is_real):
        self.row = frame['row_idx'].to_numpy(np.int64)
        self.L = build_labels(frame, is_real)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[self.row[i]]
        x[1] = M_MM[self.row[i]]
        L = self.L
        return (torch.from_numpy(x), torch.from_numpy(L['lam'][i]),
                torch.from_numpy(L['lat6'][i]), torch.tensor(L['latm'][i]),
                torch.tensor(L['sg'][i]), torch.tensor(L['sgm'][i]),
                torch.tensor(L['sys_'][i]), torch.tensor(L['sysm'][i]),
                torch.from_numpy(L['el'][i]), torch.tensor(L['elm'][i]),
                torch.tensor(L['vol'][i]), torch.tensor(L['volm'][i]))

class MixedLoader:
    def __init__(self, real_ds, syn_ds, batch, frac):
        self.real, self.syn, self.batch, self.frac = real_ds, syn_ds, batch, frac
        self.n_syn = max(1, int(batch * frac))
        self.n_real = max(1, batch - self.n_syn)
        self.steps = max(1, len(real_ds) // self.n_real)

    def __len__(self):
        return self.steps

    def __iter__(self):
        real_idx = np.random.permutation(len(self.real))
        for b in range(self.steps):
            r = real_idx[b * self.n_real:(b + 1) * self.n_real]
            s = np.random.randint(0, len(self.syn), size=self.n_syn)
            items = [self.real[i] for i in r] + [self.syn[j] for j in s]
            yield [torch.stack(t) for t in zip(*items)]

In [ ]:
# ---------- модель v2 ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)
        if cin == cout and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                                      nn.GroupNorm(8, cout))

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))
        return F.gelu(h + self.skip(x))

class XRDNetV2(nn.Module):
    def __init__(self, n_el, w=3):
        super().__init__()
        c = [32 * w, 48 * w, 64 * w, 96 * w, 128 * w, 192 * w, 256 * w]
        self.stem = nn.Sequential(nn.Conv1d(2, c[0], 15, padding=7, bias=False),
                                  nn.GroupNorm(8, c[0]), nn.GELU())
        self.blocks = nn.Sequential(*[ResBlock(c[i], c[i + 1], stride=2)
                                      for i in range(6)])
        self.lam_mlp = nn.Sequential(nn.Linear(3, 16 * w), nn.GELU(), nn.Linear(16 * w, 16 * w))
        self.trunk = nn.Sequential(nn.Linear(c[-1] + 16 * w, 512 * w), nn.GELU(),
                                   nn.Linear(512 * w, 512 * w), nn.GELU())
        self.head_lat = nn.Linear(512 * w, 6)
        self.head_vol = nn.Linear(512 * w, 1)
        self.head_sg = nn.Linear(512 * w, 230)
        self.head_sys = nn.Linear(512 * w, 7)
        self.head_el = nn.Linear(512 * w, n_el)

    def forward(self, x, lam):
        f = self.stem(x)
        f = self.blocks(f)
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = torch.cat([pooled, self.lam_mlp(lam)], dim=1)
        z = self.trunk(z)
        return dict(lat=self.head_lat(z), vol=self.head_vol(z).squeeze(-1),
                    sg=self.head_sg(z), sys=self.head_sys(z), el=self.head_el(z))

def load_pretrained():
    m = XRDNetV2(len(VOCAB), w=W).to(DEVICE)
    sd = torch.load(CKPT_PRE, map_location=DEVICE, weights_only=True)
    m.load_state_dict(sd)
    return m

print('модель v2 готова к загрузке чекпойнта')

модель v2 готова к загрузке чекпойнта


In [ ]:
# ---------- лоссы, метрики ----------
def masked_l1(pred, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return pred.new_zeros(())
    return F.smooth_l1_loss(pred[m], tgt[m])

def masked_ce(logits, tgt, mask):
    m = mask > 0
    if m.sum() == 0:
        return logits.new_zeros(())
    return F.cross_entropy(logits[m], tgt[m].long())

def compute_losses(out, batch):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch
    m = latm > 0
    if m.sum() > 0:
        l_len = F.smooth_l1_loss(out['lat'][m][:, :3], lat[m][:, :3])
        l_ang = F.smooth_l1_loss(out['lat'][m][:, 3:], lat[m][:, 3:])
    else:
        l_len = l_ang = out['lat'].new_zeros(())
    if elm.sum() > 0:
        me = (elm > 0)
        loss_el = F.binary_cross_entropy_with_logits(out['el'][me], el[me])
    else:
        loss_el = out['el'].new_zeros(())
    losses = dict(lat=l_len, ang=l_ang,
                  vol=masked_l1(out['vol'], vol, volm),
                  sg=masked_ce(out['sg'], sg, sgm),
                  sys=masked_ce(out['sys'], sys_, sysm),
                  el=loss_el)
    weighted = (W_LAT * l_len + W_ANG * l_ang + W_VOL * losses['vol']
                + W_SG * losses['sg'] + W_SYS * losses['sys'] + W_EL * loss_el)
    return losses, weighted

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    rows = []
    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=True) for b in batch]
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(batch[0], batch[1])
        _, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm = batch
        lat_pred = out['lat'].float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_pred[:, :3] = np.exp(lat_pred[:, :3])
        lat_true = lat.cpu().numpy() * LAT_STD + LAT_MEAN
        lat_true[:, :3] = np.exp(lat_true[:, :3])
        for i in range(len(lat)):
            rows.append(dict(
                latm=float(latm[i]), sgm=float(sgm[i]), sysm=float(sysm[i]),
                elm=float(elm[i]),
                sg_ok=float(sgm[i] > 0 and out['sg'][i].argmax().item() == sg[i].item()),
                sg_top5=float(sgm[i] > 0 and sg[i].item() in out['sg'][i].topk(5).indices.tolist()),
                sys_ok=float(sysm[i] > 0 and out['sys'][i].argmax().item() == sys_[i].item()),
                mae_a=abs(lat_pred[i, 0] - lat_true[i, 0]) if latm[i] > 0 else np.nan,
                mae_b=abs(lat_pred[i, 1] - lat_true[i, 1]) if latm[i] > 0 else np.nan,
                mae_c=abs(lat_pred[i, 2] - lat_true[i, 2]) if latm[i] > 0 else np.nan,
                mae_ang=float(np.abs(lat_pred[i, 3:] - lat_true[i, 3:]).mean()) if latm[i] > 0 else np.nan,
                pred_el=frozenset(np.where(torch.sigmoid(out['el'][i]).cpu().numpy() > 0.5)[0]),
                true_el=frozenset(np.where(el[i].cpu().numpy() > 0.5)[0]) if elm[i] > 0 else frozenset(),
                elm_f=float(elm[i]),
            ))
    model.train()
    d = pd.DataFrame(rows)
    res = {}
    v = d[d['sgm'] > 0]
    res['sg_acc'] = v['sg_ok'].mean() if len(v) else np.nan
    res['sg_top5'] = v['sg_top5'].mean() if len(v) else np.nan
    res['sg_n'] = len(v)
    v = d[d['sysm'] > 0]
    res['sys_acc'] = v['sys_ok'].mean() if len(v) else np.nan
    v = d[d['latm'] > 0]
    if len(v):
        res['mae_a'] = v['mae_a'].mean()
        res['mae_a_med'] = v['mae_a'].median()
        res['mae_ang'] = v['mae_ang'].mean()
    v = d[(d['elm'] > 0)]
    if len(v):
        tp = sum(len(r['pred_el'] & r['true_el']) for _, r in v.iterrows())
        fp = sum(len(r['pred_el'] - r['true_el']) for _, r in v.iterrows())
        fn = sum(len(r['true_el'] - r['pred_el']) for _, r in v.iterrows())
        p = tp / max(tp + fp, 1)
        rc = tp / max(tp + fn, 1)
        res['el_f1_micro'] = 2 * p * rc / max(p + rc, 1e-9)
        res['el_exact'] = float((v['pred_el'] == v['true_el']).mean())
    res['n'] = len(d)
    return res, d

In [ ]:
# ---------- обучение одного фолда ----------
def train_fold(train_frame, val_frame, epochs, verbose=True):
    model = load_pretrained()
    backbone = [p for n, p in model.named_parameters()
                if not n.startswith(('head_', 'trunk', 'lam_mlp'))]
    heads = [p for n, p in model.named_parameters()
             if n.startswith(('head_', 'trunk', 'lam_mlp'))]
    opt = torch.optim.AdamW([
        {'params': backbone, 'lr': HP['lr_backbone']},
        {'params': heads, 'lr': HP['lr_head']},
    ], weight_decay=HP['wd'])
    scaler = torch.amp.GradScaler('cuda')

    real_ds = SpecDS(train_frame, True)
    syn_ds = SpecDS(syn, False)
    loader = MixedLoader(real_ds, syn_ds, HP['batch'], HP['replay_frac'])
    val_loader = DataLoader(SpecDS(val_frame, True), batch_size=HP['batch'],
                            shuffle=False, pin_memory=True)

    steps_total = epochs * len(loader)
    warmup = max(1, int(steps_total * HP['warmup_frac']))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / warmup if s < warmup else
        0.5 * (1 + math.cos(math.pi * min((s - warmup) / max(steps_total - warmup, 1), 1)))))

    best_score, best_epoch, best_state = -1.0, 0, None
    for epoch in range(epochs):
        model.train()
        for batch in loader:
            batch = [b.to(DEVICE, non_blocking=True) for b in batch]
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(batch[0], batch[1])
                _, total = compute_losses(out, batch)
            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()
        m, _ = evaluate(model, val_loader)
        score = ((m.get('sys_acc') or 0) + (m.get('el_f1_micro') or 0)
                 + 0.5 * (m.get('sg_acc') or 0))
        if verbose:
            print(f"    epoch {epoch+1:2d} | sys {m.get('sys_acc', float('nan')):.3f} "
                  f"sg {m.get('sg_acc', float('nan')):.3f} "
                  f"el {m.get('el_f1_micro', float('nan')):.3f} "
                  f"mae_a {m.get('mae_a', float('nan')):.2f}", flush=True)
        if score > best_score:
            best_score, best_epoch = score, epoch + 1
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch, best_score

In [ ]:
# ---------- 5-fold group CV ----------
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=HP['n_folds'])
folds = list(gkf.split(ft, groups=ft['conn_key']))
print('фолды собраны (группировка по conn_key, утечки исключены конструктивно)')
for k, (tr_i, va_i) in enumerate(folds):
    print(f'  fold {k+1}: train {len(tr_i)} / val {len(va_i)}')

fold_metrics = []
best_epochs = []
zero_shot_metrics = []
t0 = time.time()

folds_to_run = folds[:1] if MODE == 'quick' else folds
for k, (tr_i, va_i) in enumerate(folds_to_run):
    print(f'\n===== FOLD {k+1}/{len(folds_to_run)} =====')
    train_frame = ft.iloc[tr_i].reset_index(drop=True)
    val_frame = ft.iloc[va_i].reset_index(drop=True)

    # zero-shot на этом вале
    zs_model = load_pretrained()
    zs_loader = DataLoader(SpecDS(val_frame, True), batch_size=HP['batch'],
                            shuffle=False, pin_memory=True)
    zs_m, _ = evaluate(zs_model, zs_loader)
    zero_shot_metrics.append(zs_m)
    print(f"  zero-shot: sys {zs_m.get('sys_acc', float('nan')):.3f} "
          f"el {zs_m.get('el_f1_micro', float('nan')):.3f} "
          f"mae_a {zs_m.get('mae_a', float('nan')):.2f}")
    del zs_model
    torch.cuda.empty_cache()

    model, best_ep, _ = train_fold(train_frame, val_frame, HP['epochs'])
    best_epochs.append(best_ep)
    m, _ = evaluate(model, DataLoader(SpecDS(val_frame, True), batch_size=HP['batch'],
                                      shuffle=False, pin_memory=True))
    fold_metrics.append(m)
    print(f"  best epoch {best_ep} | итог: sys {m.get('sys_acc', float('nan')):.3f} "
          f"sg {m.get('sg_acc', float('nan')):.3f} el {m.get('el_f1_micro', float('nan')):.3f} "
          f"mae_a {m.get('mae_a', float('nan')):.2f}")
    del model
    torch.cuda.empty_cache()

print(f'\nCV заняла {(time.time()-t0)/60:.1f} мин')

фолды собраны (группировка по conn_key, утечки исключены конструктивно)
  fold 1: train 2780 / val 695
  fold 2: train 2780 / val 695
  fold 3: train 2780 / val 695
  fold 4: train 2780 / val 695
  fold 5: train 2780 / val 695

===== FOLD 1/5 =====
  zero-shot: sys 0.323 el 0.166 mae_a 3.41
    epoch  1 | sys 0.363 sg 0.044 el 0.166 mae_a 3.38
    epoch  2 | sys 0.419 sg 0.104 el 0.173 mae_a 3.33
    epoch  3 | sys 0.465 sg 0.156 el 0.186 mae_a 3.27
    epoch  4 | sys 0.488 sg 0.178 el 0.206 mae_a 3.23
    epoch  5 | sys 0.496 sg 0.222 el 0.230 mae_a 3.18
    epoch  6 | sys 0.527 sg 0.252 el 0.253 mae_a 3.12
    epoch  7 | sys 0.548 sg 0.259 el 0.281 mae_a 3.08
    epoch  8 | sys 0.556 sg 0.274 el 0.308 mae_a 3.03
    epoch  9 | sys 0.571 sg 0.289 el 0.322 mae_a 2.99
    epoch 10 | sys 0.575 sg 0.289 el 0.330 mae_a 2.96
    epoch 11 | sys 0.585 sg 0.304 el 0.347 mae_a 2.92
    epoch 12 | sys 0.579 sg 0.311 el 0.360 mae_a 2.88
    epoch 13 | sys 0.577 sg 0.304 el 0.369 mae_a 2.85
    ep

In [ ]:
# ---------- сводка CV: среднее ± std по фолдам ----------
keys = ['sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro', 'el_exact',
        'mae_a', 'mae_a_med', 'mae_ang']
rows = []
for k in keys:
    zs = [m.get(k) for m in zero_shot_metrics]
    ft_ = [m.get(k) for m in fold_metrics]
    zs_v = [v for v in zs if v == v]
    ft_v = [v for v in ft_ if v == v]
    rows.append(dict(
        metric=k,
        zero_shot=f'{np.mean(zs_v):.3f}' if zs_v else '-',
        after_ft=f'{np.mean(ft_v):.3f}' if ft_v else '-',
        std=f'{np.std(ft_v):.3f}' if len(ft_v) > 1 else '-',
        folds=len(ft_v),
    ))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()
if len(fold_metrics) > 1:
    print('медиана лучших эпох фолдов:', int(np.median(best_epochs)),
          '| лучшие эпохи:', best_epochs)
summary.to_csv(OUT / 'ft_v2_cv_summary.csv', index=False)

     metric zero_shot after_ft   std  folds
    sys_acc     0.345    0.630 0.022      5
     sg_acc     0.069    0.509 0.212      5
    sg_top5     0.143    0.676 0.238      5
el_f1_micro     0.181    0.501 0.048      5
   el_exact     0.002    0.179 0.041      5
      mae_a     3.466    2.662 0.170      5
  mae_a_med     2.179    1.523 0.138      5
    mae_ang     6.034    4.574 0.133      5

медиана лучших эпох фолдов: 41 | лучшие эпохи: [45, 41, 39, 33, 48]


In [ ]:
# ---------- финальная модель: весь пул, эпох = медиана лучших ----------
if MODE == 'cv':
    n_ep = int(np.clip(np.median(best_epochs), 5, HP['epochs']))
    print(f'финальное обучение: весь пул {len(ft)}, {n_ep} эпох')
    # валидируемся на synthetic-val, чтобы ловить переобучение в процессе
    syn_val = syn.sample(min(3000, len(syn)), random_state=SEED)
    model, _, _ = train_fold(ft.reset_index(drop=True), syn_val, n_ep, verbose=True)
    torch.save(model.state_dict(), CKPT_DIR / 'ft_v2_final.pt')
    print('сохранено: checkpoints/ft_v2_final.pt')
else:
    print('MODE=quick: финальная модель не обучается')

финальное обучение: весь пул 3475, 41 эпох
    epoch  1 | sys 0.614 sg 0.455 el 0.674 mae_a 2.73
    epoch  2 | sys 0.612 sg 0.454 el 0.672 mae_a 2.72
    epoch  3 | sys 0.608 sg 0.446 el 0.672 mae_a 2.73
    epoch  4 | sys 0.609 sg 0.445 el 0.673 mae_a 2.73
    epoch  5 | sys 0.610 sg 0.447 el 0.673 mae_a 2.73
    epoch  6 | sys 0.612 sg 0.448 el 0.673 mae_a 2.73
    epoch  7 | sys 0.613 sg 0.451 el 0.671 mae_a 2.73
    epoch  8 | sys 0.610 sg 0.448 el 0.671 mae_a 2.73
    epoch  9 | sys 0.612 sg 0.450 el 0.672 mae_a 2.73
    epoch 10 | sys 0.613 sg 0.449 el 0.671 mae_a 2.73
    epoch 11 | sys 0.611 sg 0.449 el 0.671 mae_a 2.73
    epoch 12 | sys 0.612 sg 0.451 el 0.670 mae_a 2.73
    epoch 13 | sys 0.612 sg 0.451 el 0.670 mae_a 2.74
    epoch 14 | sys 0.615 sg 0.455 el 0.669 mae_a 2.74
    epoch 15 | sys 0.612 sg 0.453 el 0.670 mae_a 2.74
    epoch 16 | sys 0.618 sg 0.455 el 0.669 mae_a 2.73
    epoch 17 | sys 0.615 sg 0.455 el 0.670 mae_a 2.74
    epoch 18 | sys 0.616 sg 0.454 el 0.

- **сводка CV** — основная таблица: `zero_shot` vs `after_ft` ± std по 5 фолдам;
- `sg_acc` оценивается только на opXRD-строках с SG (в каждом фолде их ~100, и доминируют несколько SG-классов — интерпретировать как «точность на знакомых SG», не как общую точность 230-классовой задачи);
- финальная модель `ft_v2_final.pt` обучена на всех 3 475 строках — её метрики = CV-сводка (с лёгким оптимизмом, т.к. данных чуть больше).